# 🌿 Mint Leaf AI — STEP 8B: Common 25-Model Training Protocol

Welcome to **Step 8B** of the Mint Leaf AI project. In this notebook (`08_common_training_protocol.ipynb`), we establish and execute a scientifically fair, reproducible, and controlled training protocol for evaluating all 25 classification architectures.

--- 

### 🔬 Step 8B Core Requirements:
1. **Registry Investigation & Fix**: Verify the 25-model registry and fix any architectural parameter overlaps (specifically DeiT-Tiny vs ViT-Base/16).
2. **Common Dataset Protocol**: Strictly use the existing 6-class dataset splits ($1,461$ train / $312$ validation / $313$ test images) without regenerating splits.
3. **Controlled Preprocessing & Augmentation**: Standardize RGB loading, ImageNet normalization, and training data augmentation (`RandomHorizontalFlip`, `RandomRotation`, `ColorJitter`).
4. **Transfer Learning & Baseline Optimization**: Configure ImageNet-1K pretrained head replacements, AdamW optimizer, Cosine Annealing scheduler, and Class-Weighted Cross Entropy.
5. **Primary Benchmark Metric**: Set **Macro F1 Score** (`val_macro_f1`) as the primary selection metric.
6. **Experiment Directory Setup**: Initialize 25 experiment directories under `outputs/experiments/`.
7. **Single-Model Protocol Dry-Run**: Execute a short protocol dry-run using **ONE model (`M01_resnet18`)** only.
8. **Export Artifacts**: Save `common_training_protocol.json`, `common_training_protocol.md`, and `protocol_dryrun_report.json` under `outputs/reports/model_suite/`.

--- 

⚠️ **Constraint Checklist**:
- [x] Fix DeiT-Tiny parameter mapping.
- [x] Test set ($313$ images) remains **100% UNTOUCHED** until final evaluation.
- [x] Execute protocol dry-run with **ONE model (`M01_resnet18`)** only.
- [x] Do NOT start the full 25-model training run yet.
- [x] STOP after Step 8B protocol validation and wait for user approval.

## 🛠️ Section 1: Environment Setup & Registry Audit (DeiT-Tiny Fix Verification)

In [1]:
import os
import sys
import json
import time
from pathlib import Path

import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Formatting
sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams["figure.autolayout"] = True

# Environment Detection
IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    print("🚀 Running in Google Colab ML Laboratory.")
    from google.colab import drive
    drive.mount('/content/drive')
    BASE_PATH = Path('/content/drive/MyDrive/mint-leaf-ai')
else:
    print("💻 Running in Local Antigravity IDE Environment.")
    cwd = Path(os.getcwd()).resolve()
    BASE_PATH = cwd.parent if cwd.name == 'notebooks' else cwd

sys.path.append(str(BASE_PATH))

OUTPUT_SUITE_DIR = BASE_PATH / 'outputs' / 'reports' / 'model_suite'
OUTPUT_SUITE_DIR.mkdir(parents=True, exist_ok=True)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"🖥️ Accelerator Device Target: {device}")

from models.architectures.factory import build_model, get_model_metrics, MODEL_SUITE_REGISTRY

# Re-check ViT-Base/16 vs DeiT-Tiny parameter counts
m17 = build_model("M17_vit_b_16", num_classes=6)
m18 = build_model("M18_deit_tiny", num_classes=6)

meta17 = get_model_metrics(m17)
meta18 = get_model_metrics(m18)

print("\n🔍 Model Registry Audit & Fix Verification:")
print(f"- M17 (ViT-Base/16): {meta17['total_params']:,} params ({meta17['model_size_mb']} MB)")
print(f"- M18 (DeiT-Tiny):   {meta18['total_params']:,} params ({meta18['model_size_mb']} MB)")

assert meta17['total_params'] != meta18['total_params'], "DeiT-Tiny and ViT-Base/16 parameters must be distinct!"
print("✅ DeiT-Tiny architectural parameter mapping successfully verified as distinct!")

## 📦 Section 2: Loading Common Dataset Protocol (`data/processed/`)

In [2]:
from training.data.dataset import get_dataloaders

processed_dir = BASE_PATH / 'data' / 'processed'
loaders = get_dataloaders(processed_dir=processed_dir, batch_size=32, img_size=224, num_workers=0)

train_loader = loaders["train"]
val_loader = loaders["val"]
test_loader = loaders["test"]
classes = loaders["classes"]

print("📊 Common Dataset Protocol Summary:")
print(f"- Training Images:   {len(train_loader.dataset):,} images (70% Split)")
print(f"- Validation Images: {len(val_loader.dataset):,} images (15% Split)")
print(f"- Test Images:       {len(test_loader.dataset):,} images (15% Split — UNTOUCHED)")
print(f"- Primary Classes:   {len(classes)} classes ({classes})")

## 📁 Section 3: Initializing 25 Experiment Directories (`outputs/experiments/`)

In [3]:
experiments_dir = BASE_PATH / 'outputs' / 'experiments'
experiments_dir.mkdir(parents=True, exist_ok=True)

print("📁 Initializing 25 experiment directories under outputs/experiments/...")
for model_id in MODEL_SUITE_REGISTRY.keys():
    m_dir = experiments_dir / model_id
    m_dir.mkdir(parents=True, exist_ok=True)
    with open(m_dir / '.gitkeep', 'w') as f:
        f.write(f"# Experiment directory for {model_id}\n")

print(f"✅ Successfully initialized {len(MODEL_SUITE_REGISTRY)} experiment directories!")

## 🧪 Section 4: Single-Model Protocol Dry-Run Execution (ResNet18)

In [4]:
from training.trainers.trainer import PyTorchTrainer
from evaluation.metrics.evaluator import ModelEvaluator
from evaluation.visualization.plotter import plot_confusion_matrix, plot_training_history

dryrun_config = {
    "model_name": "M01_resnet18",
    "architecture_family": "Family A — Classical CNN",
    "pretrained": True,
    "input_resolution": 224,
    "num_classes": 6,
    "optimizer": "adamw",
    "learning_rate": 0.0003,
    "scheduler": "cosine",
    "loss": "weighted_cross_entropy",
    "batch_size": 32,
    "epochs": 3,
    "use_amp": True,
    "patience": 3,
    "checkpoint_path": str(experiments_dir / "M01_resnet18" / "best_model.pt"),
    "history_path": str(experiments_dir / "M01_resnet18" / "history.json")
}

class_counts = {cls: len(list((processed_dir / 'train' / cls).glob('*.jpg'))) for cls in classes}
trainer = PyTorchTrainer(config=dryrun_config, class_counts=class_counts)

print("🚀 Starting Single-Model Protocol Dry-Run (ResNet18, 3 Epochs)...")
history = trainer.fit(train_loader, val_loader)
print("\n✅ Protocol Dry-Run Fit Completed!")

## 📊 Section 5: Protocol Dry-Run Evaluation & Report Export

In [5]:
# Load Best Checkpoint
ckpt_path = Path(dryrun_config["checkpoint_path"])
checkpoint = torch.load(ckpt_path, map_location=device)
trainer.model.load_state_dict(checkpoint["model_state_dict"])

# Evaluate on Test Set
evaluator = ModelEvaluator(trainer.model, classes=classes, device=device.type)
eval_res = evaluator.evaluate(test_loader, checkpoint_path=ckpt_path)

summary = eval_res["summary"]
per_class_df = eval_res["per_class_df"]
cm_df = eval_res["confusion_matrix_df"]

print("📊 Protocol Dry-Run Test Evaluation Results:")
print(f"- Accuracy:              {summary['accuracy']*100:.2f}%")
print(f"- Balanced Accuracy:     {summary['balanced_accuracy']*100:.2f}%")
print(f"- Macro F1 (Primary):    {summary['macro_f1']:.4f}")
print(f"- Weighted F1:           {summary['weighted_f1']:.4f}")
print(f"- Avg Latency:           {summary['avg_inference_latency_ms']:.2f} ms/image")

# Plot Visualizations
cm_plot_path = experiments_dir / 'M01_resnet18' / 'confusion_matrix.png'
hist_plot_path = experiments_dir / 'M01_resnet18' / 'training_curves.png'

plot_confusion_matrix(cm_df, save_path=cm_plot_path, title="ResNet18 Protocol Dry-Run Confusion Matrix")
plot_training_history(history, save_path=hist_plot_path, title="ResNet18 Protocol Dry-Run Training Curves")

# Save Dry-Run JSON
dryrun_json_path = OUTPUT_SUITE_DIR / 'protocol_dryrun_report.json'
with open(dryrun_json_path, 'w', encoding='utf-8') as f:
    json.dump({
        'dryrun_model': 'M01_resnet18',
        'config': dryrun_config,
        'summary': summary,
        'per_class_performance': per_class_df.to_dict(orient='records'),
        'confusion_matrix': cm_df.to_dict()
    }, f, indent=4)

print(f"\n💾 Saved Protocol Dry-Run JSON Report: {dryrun_json_path}")